# WAXAL Amharic retrain — Kaggle (free GPU)

Fine-tunes `badrex/Ethio-ASR-amharic` (the production checkpoint) on WAXAL
Amharic ASR shards, **freezing the conformer trunk and training the CTC head**
by default — the same recipe as `tools/retrain/` in the repo.

**The gate (honest):** this run never replaces the shipped model automatically.
The gate cell runs `tools/retrain/04_eval_wer.py` on a **held-out slice** the
production model has never seen:

  * `current`   = `badrex/Ethio-ASR-amharic` (the model shipped today)
  * `retrained` = what this run produces

If **retrained avg WER <= current avg WER** on the same slice → KEEP, then
export CTranslate2 int8 for the product. If it regresses → REJECT; the current
model stays untouched and you just tweak the config and re-run.

Numerical note: this gate measures WER on WAXAL held-out rows (same-domain
Amharic speech), NOT the repo's honest fixture set. The honest-set WER for the
shipped model is **0.227** (40 real clips, `tools/test/wer.py`). Re-run that
fixture set back on the Mac after the CT2 export as a final sanity check — the
gate here is the *relative* current-vs-retrained comparison.

## Config — edit this first

| constant | meaning |
|---|---|
| `N_SHARDS`       | train shards to pull (1 shard ≈ 490 MB ≈ ~1000 rows) |
| `MAX_TOTAL_BYTES`| hard cap on fetched bytes (None = no cap) |
| `DEV_ROWS`       | last N manifest rows held out as the WER gate slice |
| `EPOCHS` `BATCH` `LR` | fine-tune hyperparameters |
| `MAX_STEPS`      | hard cap on optimizer steps (None = run until epochs done) |
| `MAX_TRAIN_ROWS` | cap rows used for training (None = use all fetched) |
| `FREEZE`         | True: train CTC head only (fast, safe). False: full fine-tune |
| `HF_MODEL`       | production checkpoint to start from |

In [ ]:
CFG = dict(
    N_SHARDS        = 3,          # 3 shards ≈ ~3000 rows, hours on a T4
    MAX_TOTAL_BYTES = None,       # e.g. "3G" to budget; None = no cap
    DEV_ROWS        = 150,        # held-out gate slice
    EPOCHS          = 3,
    BATCH           = 8,
    LR              = 3e-4,
    MAX_STEPS       = None,       # e.g. 2500 for a quick budgeted run
    MAX_TRAIN_ROWS  = None,
    FREEZE          = True,       # recommended: head-only
    HF_MODEL        = "badrex/Ethio-ASR-amharic",
)
WX    = "/kaggle/working/waxal"          # parquet + wavs + manifests
OUT   = "/kaggle/working/model-retrained"
CT2   = "/kaggle/working/model-ct2-int8-retrain"
ROOT  = "/kaggle/working/amharic-caption"
import os; os.makedirs(WX, exist_ok=True)

In [ ]:
import subprocess, sys
def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=False, text=True, **kw)
    if r.returncode:
        print(f"[fail] rc={r.returncode}: {cmd}")
        raise SystemExit(r.returncode)
    return r

# 1) clone the (public) repo so the existing retrain pipeline runs as-is
if not os.path.isdir(ROOT):
    run(["git", "clone", "--depth", "1",
         "https://github.com/kaleb21-19/amharic_caption", ROOT])
print("repo ready")

# 2) top up packages (torch/torchaudio/pyarrow/pandas ship with the image)
pip = [sys.executable, "-m", "pip", "install", "-q", "--no-input",
       "transformers>=4.52", "soundfile", "ctranslate2"]
for _ in range(3):
    r = subprocess.run(pip, capture_output=True, text=True)
    if r.returncode == 0:
        break
    print(r.stderr[-500:])
else:
    raise SystemExit("pip install failed 3x")

import torch, transformers
print("torch cuda =", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("transformers =", transformers.__version__)

In [ ]:
# 3) FETCH — download N WAXAL Amharic ASR shards (chunked, resumes).
#    This is the same script the repo uses; fast on Kaggle's network.
cmd = ["python3", f"{ROOT}/tools/retrain/01_fetch_waxal.py", "--out", WX,
       "--max-shards", str(CFG["N_SHARDS"])]
if CFG["MAX_TOTAL_BYTES"]:
    cmd += ["--max-total-bytes", str(CFG["MAX_TOTAL_BYTES"])]
run(cmd)
import glob
shards = sorted(glob.glob(f"{WX}/*.parquet"))
tot = sum(os.path.getsize(p) for p in shards)
print(f"[ok] {len(shards)} shards, {tot/1e9:.2f} GB")

In [ ]:
# 4) PREP — parquet -> 16 kHz wavs + manifest.tsv, then carve the held-out
#    slice exactly like run_all.sh does (dev = LAST rows of the manifest).
run(["python3", f"{ROOT}/tools/retrain/02_prep_waxal.py",
     "--shards", WX, "--wavs", f"{WX}/wavs",
     "--manifest", f"{WX}/manifest.tsv"])

man = f"{WX}/manifest.tsv"; dev = f"{WX}/dev.tsv"
rows = open(man, encoding="utf-8").read().splitlines()
n = min(CFG["DEV_ROWS"], len(rows))
open(dev, "w", encoding="utf-8").write("\n".join(rows[-n:]) + "\n")
print(f"[split] train={len(rows)-n} dev={n} -> {dev}")

In [ ]:
# 5) sanity peek at the manifest
for line in open(f"{WX}/manifest.tsv", encoding="utf-8").read().splitlines()[:3]:
    p, _s, t = line.split("\t", 2)
    print(os.path.basename(p), "|", t[:60])

In [ ]:
# 6) FINE-TUNE — CTC head on WAXAL (encoder frozen by default).
#    Device auto-selects CUDA; loss is computed on CPU so even a regression
#    to CPU/MPS would work, just slower.
cmd = ["python3", f"{ROOT}/tools/retrain/03_finetune_waxal.py",
       "--manifest", f"{WX}/manifest.tsv",
       "--src", CFG["HF_MODEL"],
       "--out", OUT,
       "--epochs", str(CFG["EPOCHS"]),
       "--batch-size", str(CFG["BATCH"]),
       "--lr", str(CFG["LR"])]
if CFG["FREEZE"]:    cmd += ["--freeze-encoder"]
if CFG["MAX_STEPS"]:   cmd += ["--max-steps", str(CFG["MAX_STEPS"])]
if CFG["MAX_TRAIN_ROWS"]: cmd += ["--max-train-rows", str(CFG["MAX_TRAIN_ROWS"])]
run(cmd)

In [ ]:
# 7) WER GATE — current vs retrained on the held-out slice.
#    exit 0 = KEEP (retrained <= current), 1 = REJECT. Nothing else ran a
#    model; this decides automatically.
dev = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
r = subprocess.run(["python3", f"{ROOT}/tools/retrain/04_eval_wer.py",
                    "--manifest", f"{WX}/dev.tsv",
                    "--current", CFG["HF_MODEL"],
                    "--candidate", OUT, "--device", dev],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-2000:] if r.returncode else "")
VERDICT = "KEEP" if r.returncode == 0 else "REJECT"
print("\n>>> GATE VERDICT:", VERDICT)

## Reading the gate

* **KEEP (exit 0)** — the retrained model is at least as good as the shipped
  one on the held-out slice → continue to the export cell below.
* **REJECT (exit 1)** — it regressed. The shipped model is untouched; the
  current run is already a useful negative result. Try more shards, more
  epochs, unfreezing (`FREEZE=False`), a different `LR`/`BATCH`, or MUSAN
  robustness data, then run cells 3–7 again. Do **not** export a REJECT.
Either way the honest fixture-set number (WER **0.227** on 40 real clips) is
the final word back on the Mac — this gate is only the relative comparison.

In [ ]:
# 8) EXPORT — CTranslate2 int8, the exact layout the product build packages.
import shutil, glob
assert os.path.isdir(OUT), "train first"
env = dict(os.environ, MODEL_SRC=OUT, MODEL_DST=CT2)
run(["bash", f"{ROOT}/tools/make_model_ct2_int8.sh"], env=env)

# bundle the HF checkpoint + ct2 int8 + gate summary for download
summary = f"\n".join([l for l in r.stdout.splitlines() if "[result]" in l])
open("/kaggle/working/SUMMARY.txt", "w", encoding="utf-8").write(
    f"gate verdict: {VERDICT}\n{summary}\n")
for what, src in (["model-retrained", OUT], ["model-ct2-int8-retrain", CT2],
                  ["SUMMARY.txt", "/kaggle/working/SUMMARY.txt"]):
    dst = f"/kaggle/working/bundle/{what}"
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
shutil.make_archive("/kaggle/working/retrain_output", "zip",
                    "/kaggle/working/bundle")
import os
print("[ok] /kaggle/working/retrain_output.zip",
      f"{os.path.getsize('/kaggle/working/retrain_output.zip')/1e6:.0f} MB")

## Installing the result back on the Mac

If (and only if) the gate said **KEEP**: download `retrain_output.zip` from the
notebook Output pane, unzip it, and on the repo — **one command**:

```bash
bash tools/retrain/install_retrained.sh /path/to/unzipped/retrain_output
```

It refuses a REJECT bundle, backs up `tools/stage/model-ct2-int8` to
`model-ct2-int8.prev`, swaps in the retrained int8 model, re-scores old vs new
on `tools/stage/waxal/holdout.tsv` (the same `04_eval_wer.py` as the gate) and
auto-reverts on regression, then rebuilds mac-arm64 + mac-x64 + win-x64
(~25-30 min). Rollback: `mv tools/stage/model-ct2-int8.prev tools/stage/model-ct2-int8`.

Then re-run the honest fixture set (same 40 real clips that gave 0.227). Keep
the swap only if the honest WER is at least as good as 0.227 — a WAXAL-dev win
is encouraging but the real measurements are what ship.

The model outputs are gitignored; nothing here needs to be committed. This
exercise costs nothing (free Kaggle GPU session).